In [123]:
from scipy.integrate import quad
from math import exp, inf, pi, erf
import numpy as np
import scipy.stats as st
import scipy.optimize as opt

In [124]:
def gamma(a):
    return quad(lambda x: x ** (a - 1) * exp(-x), 0, inf)[0]

def p(n, x):
    return (1/2) ** (n/2) / gamma(n/2) * x ** (n/2 - 1) * exp(-x/2)

def X_2(N, a, b):
    return quad(lambda x: p(N, x), a, b)[0]

In [125]:
result = []

Пункт а), критерий хи квадрат

In [126]:
result_a_X_2 = {"название:": "Равномерное распределение, критерий хи квадрат",
                "оценка дельты =": 16.4,
                "p-value =": X_2(9, 16.4, inf),
                "решение для H0:": "Нет веских оснований отвергнуть H0"}
result.append(result_a_X_2)

Пункт а), критерий Колмагорова

In [127]:
def K(x):
    res = 0
    for k in range(1, 10000):
        res += (-1) ** k * exp(-2 * k ** 2 * x ** 2)
    return 1 + 2 * res

In [128]:
def get_delta_estimation_for_K(x_n, n, k):
    unit = np.ones(k)
    f_uni = np.array([unit[:i].sum() for i in range(len(unit))]) / 10
    f_emp = np.array([x_n[:i].sum() for i in range(len(x_n) + 1)]) / n

    max_diff = max(max(abs(f_emp[i] - f_uni[i]), abs(f_emp[i + 1] - f_uni[i])) for i in range(k))
    delta_estimation = n ** 0.5 * max_diff
    return delta_estimation

In [129]:
x_n = np.array([5, 8, 6, 12, 14, 18, 11, 6, 13, 7])

n = 100
k = len(x_n)
alpha = 0.05

delta_estimation = get_delta_estimation_for_K(x_n, n, k)
p_value = 1 - K(delta_estimation)

result_a_K = {"название:": "Равномерное распределение, критерий Колмагорова",
                "оценка дельты =": delta_estimation,
                "p-value =": p_value,
                "решение для H0:": "Отвергаем H0" if p_value < alpha else "Нет веских оснований отвергнуть H0"}
result.append(result_a_K)

Параметрический бутстрап

In [130]:
k_boot = 50000
bootstrap_estimates = []
for _ in range(k_boot):
    boot_sample = st.uniform.rvs(loc=0, scale=10, size=n)
    boot_delta = get_delta_estimation_for_K(boot_sample, n, k)
    bootstrap_estimates.append(boot_delta)

bootstrap_estimates = np.array(sorted(bootstrap_estimates))
p_value = np.mean(np.array(bootstrap_estimates) >= delta_estimation) / k_boot
result_a_K_boot = {"название:": "Равномерное распределение, критерий Колмагорова, параметрический бутстрап",
                "p-value =": p_value,
                "решение для H0:": "Отвергаем H0" if p_value < alpha else "Нет веских оснований отвергнуть H0"}
result.append(result_a_K_boot)

Пункт б), вычисления для критерия хи-квадрат

In [131]:
# (-∞, 0.5], (0.5, 1.5], ..., (8.5, 9.5], (9.5, ∞)
boundaries = np.arange(0.5, 11.5, 1)

def neg_log_likelihood(params):
    mu, sigma = params
    if sigma <= 0:
        return 1e10
    
    probs = np.diff(st.norm.cdf(boundaries, loc=mu, scale=sigma))
    
    probs = np.maximum(probs, 1e-15)
    probs = probs / np.sum(probs)
    
    return -np.sum(x_n * np.log(probs))

res = opt.differential_evolution(
    neg_log_likelihood,
    bounds=[(0, 10), (0, 10)],
    maxiter=10000
)

mu_mle, sigma_mle = res.x
print(f"ОМПГ оценки: mu = {mu_mle:.4f}, sigma = {sigma_mle:.4f}")

def norm(theta_1, theta_2, x):
    return 1 / ((2 * pi) ** 0.5 * theta_2) * exp(-((x - theta_1) ** 2) / 2 / (theta_2 ** 2))

estimation_delta = 0
boundaries = [[-inf, 1], [1, 2], [2, 3], [3, 4], [4, 5], [5, 6], [6, 7], [7, 8], [8, 9], [9, inf]]
for i in range(10):
    p_i = quad(lambda x: norm(mu_mle, sigma_mle, x), *boundaries[i])
    estimation_delta += (x_n[i] - n * p_i[0]) ** 2 / n / p_i[0]
print(f"Оценка дельты: {estimation_delta:.4f}")
p_value = X_2(9, estimation_delta, inf)
print(f"p-value = {p_value:.4f}")

result_b_X_2 = {"название:": "Нормальное распределение, критерий хи квадрат",
                "оценка дельты =": estimation_delta,
                "p-value =": p_value,
                "решение для H0:": "Отвергаем H0" if p_value < alpha else "Нет веских оснований отвергнуть H0"}
result.append(result_b_X_2)

ОМПГ оценки: mu = 6.0291, sigma = 3.5018
Оценка дельты: 21.5198
p-value = 0.0105


Пункт б), критерий Колмагорова

In [132]:
def norm_cdf(x, a, b):
    return 0.5 * (1 + erf((x - a) / (np.sqrt(2) * b)))

In [133]:
def get_delta_estimation_for_K_norm(x_n, n, k, theta1, theta2):
    grid = np.arange(k)
    f_emp = np.array([x_n[:i].sum() for i in range(len(x_n) + 1)]) / n

    max_diff = max(max(abs(f_emp[i] - norm_cdf(grid[i], theta1, theta2)), 
                       abs(f_emp[i + 1] - norm_cdf(grid[i], theta1, theta2))) for i in range(k))
    delta_estimation = n ** 0.5 * max_diff
    return delta_estimation

In [134]:
delta_estimation = get_delta_estimation_for_K_norm(x_n, n, k, mu_mle, sigma_mle)
p_value = 1 - K(delta_estimation)

result_b_K = {"название:": "Нормальное распределение, критерий Колмагорова",
                "оценка дельты =": delta_estimation,
                "p-value =": p_value,
                "решение для H0:": "Отвергаем H0" if p_value < alpha else "Нет веских оснований отвергнуть H0"}
result.append(result_b_K)

Параметрический бутстрап

In [135]:
k_boot = 50000
bootstrap_estimates = []
for _ in range(k_boot):
    boot_sample = st.norm.rvs(loc=mu_mle, scale=sigma_mle, size=n)
    
    mu_mle = boot_sample.mean()
    sigma_mle = boot_sample.std(ddof=1)

    boot_delta = get_delta_estimation_for_K_norm(boot_sample, n, k, mu_mle, sigma_mle)
    bootstrap_estimates.append(boot_delta)

bootstrap_estimates = np.array(sorted(bootstrap_estimates))
p_value = np.mean(np.array(bootstrap_estimates) >= delta_estimation) / k_boot
result_b_K_boot = {"название:": "Нормальное распределение, критерий Колмагорова, параметрический бутстрап",
                "p-value =": p_value,
                "решение для H0:": "Отвергаем H0" if p_value < alpha else "Нет веских оснований отвергнуть H0"}
result.append(result_b_K_boot)

In [137]:
for elem in result:
    for key in elem:
        if type(elem[key]) == float:
            val = f"{elem[key]:.6f}"
        else:
            val = elem[key]
        print(f"{key} {val}")
    print()

название: Равномерное распределение, критерий хи квадрат
оценка дельты = 16.400000
p-value = 0.058984
решение для H0: Нет веских оснований отвергнуть H0

название: Равномерное распределение, критерий Колмагорова
оценка дельты = 1.4000000000000001
p-value = 0.039682
решение для H0: Отвергаем H0

название: Равномерное распределение, критерий Колмагорова, параметрический бутстрап
p-value = 1.99976e-05
решение для H0: Отвергаем H0

название: Нормальное распределение, критерий хи квадрат
оценка дельты = 21.519849040457387
p-value = 0.010532
решение для H0: Отвергаем H0

название: Нормальное распределение, критерий Колмагорова
оценка дельты = 2.455790512949403
p-value = 0.000012
решение для H0: Отвергаем H0

название: Нормальное распределение, критерий Колмагорова, параметрический бутстрап
p-value = 1.9992e-05
решение для H0: Отвергаем H0

